In [1]:
# from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
load_dotenv()

True

In [3]:
from dotenv import load_dotenv
import os

In [4]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
GROQ_API_KEY

'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'

In [5]:
llm = ChatGroq(  
model="llama-3.1-8b-instant",  
temperature=0.0,  
max_retries=2, 
)

In [6]:
llm.invoke("what is mcp in 100 words")

AIMessage(content="MCP stands for Master of Computer Programming. It is a professional certification offered by Microsoft to validate an individual's skills in programming and software development. The certification is designed for developers who want to demonstrate their expertise in Microsoft technologies, such as .NET, Azure, and Visual Studio.\n\nTo become an MCP, candidates must pass a series of exams that test their knowledge and skills in specific areas of programming, such as:\n\n- Programming languages (e.g., C#, Java)\n- Development frameworks (e.g., ASP.NET, Angular)\n- Database management (e.g., SQL Server)\n- Cloud computing (e.g., Azure)\n\nMCP certification is recognized globally and can enhance a developer's career prospects.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 43, 'total_tokens': 184, 'completion_time': 0.217528401, 'completion_tokens_details': None, 'prompt_time': 0.010154597, 'prompt_tokens_details': N

In [7]:
folder_path="./data/policies"

#### Docs Loading

In [8]:
# """
# 1. retrieve_hr_policy
# 2. retrieve_travel_policy
# 3. retrieve_reimbursement_policy
# 4. retrieve_it_security_policy
# 5. retrieve_ai_usage_policy
# 6. grade_context
# 7. rewrite_query
# 8. generate_grounded_answer
# 9. review_answer_grounding
# 10. ask_clarification
# """

#### Doc Loading

In [9]:
import os
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_documents(folder_path="./data/policies"):
    docs = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".md"):   # adjust extension if needed
            file_path = os.path.join(folder_path, filename)
            loader = TextLoader(file_path, encoding="utf-8")
            loaded_docs = loader.load()
            # normalize metadata for each doc
            for i, doc in enumerate(loaded_docs, start=1):
                source_path = Path(file_path)
                doc.metadata = {
                    "source_file": source_path.name,
                    "policy_domain": source_path.stem,
                    "page_number": str(i),
                }
                docs.append(doc)
    return docs


In [10]:
docs = load_documents("./data/policies")

#### Chunking

In [11]:
def split_and_set_metadata(docs, chunk_size=800, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunk_documents = text_splitter.split_documents(docs)

    chunks = []
    for i, chunk_document in enumerate(chunk_documents, start=1):
        metadata = dict(chunk_document.metadata)
        metadata["chunk_id"] = f"chunk_{i}"
        chunks.append(Document(page_content=chunk_document.page_content, metadata=metadata))
    return chunks


In [12]:

# view

chunks = split_and_set_metadata(docs)

print(f"Loaded {len(docs)} docs.")
print(f"Total chunks: {len(chunks)}")
print(chunks[0].metadata)

Loaded 5 docs.
Total chunks: 17
{'source_file': 'ai_usage_policy.md', 'policy_domain': 'ai_usage_policy', 'page_number': '1', 'chunk_id': 'chunk_1'}


In [13]:
if docs:
    print(f"Loaded {len(docs)} pages from PDF.\n")
    print("First page content:\n", docs[0].page_content[:500], "...")
    print("\nMetadata:", docs[3].metadata)
    print("\nMetadata keys:", docs[0].metadata.keys())

Loaded 5 pages from PDF.

First page content:
 # AI Usage Policy
**Policy ID:** AI-GOV-001
**Version:** 1.0
**Effective Date:** 2026-01-01
**Policy Owner:** AI Governance Committee

## 1. Purpose
This policy defines acceptable and restricted use of AI tools, including public AI tools, approved internal AI tools, customer data handling, confidential data rules, human review, and AI use case approval.

## 2. Approved AI Tools
Employees may use only company-approved AI tools for business work.

Public AI tools may be used only for low-risk task ...

Metadata: {'source_file': 'reimbursement_policy.md', 'policy_domain': 'reimbursement_policy', 'page_number': '1'}

Metadata keys: dict_keys(['source_file', 'policy_domain', 'page_number'])


In [14]:
persist_directory="./vector_store"
chunk_collection_name="policy-chunks"

In [15]:
import chromadb

In [16]:
chromadb

<module 'chromadb' from 'c:\\Training\\Assignments\\AI_Training_Batch_May_2026\\submissions\\project-build\\langgraph-application\\Mohammad-Zaid\\enterprise_policy_agentic_rag\\.venv\\Lib\\site-packages\\chromadb\\__init__.py'>

In [17]:
chromadb_client = chromadb.PersistentClient(
    path=persist_directory
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [18]:
chromadb_client

In [19]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Training\Assignments\AI_Training_Batch_May_2026\submissions\project-build\langgraph-application\Mohammad-Zaid\enterprise_policy_agentic_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
embedding_model

In [ ]:

vector_db = Chroma(collection_name=chunk_collection_name,
                   collection_metadata={"hnsw:space": "cosine"}, 
                   embedding_function=embedding_model,
                   client=chromadb_client, 
                   persist_directory=persist_directory
                   )

In [ ]:
# def add_chunks_to_vector_db(chunks, vector_db):
vector_db._collection.count()

vector_db.add_documents(
        documents=chunks,
        ids=[chunk.metadata["chunk_id"] for chunk in chunks],
        )
        

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
# Test retrieval
query = "What are the key financial highlights of Amazon in 2025?"
retrieved_chunks = retriever.invoke(query)

In [ ]:
context = "\n\n".join([
    f"Source: {chunk.metadata.get('source_file')} | "
    f"Policy: {chunk.metadata.get('policy_domain')} | "
    f"Page: {chunk.metadata.get('page_number')} | "
    f"Chunk: {chunk.metadata.get('chunk_id')}\n"
    f"{chunk.page_content}"
    for chunk in retrieved_chunks
])


In [ ]:
print(context)


In [ ]:
system_prompt = """

You are an enterprise policy assistant agent.
Use the Correct tool to Answer the user question.
Rules:
- Do not use outside knowledge.
- If the answer is not available in the context, say: "I could not find this in the provided documents."
- Cite the source file and page number or chunk ID for each key claim.
- Do not invent numbers, dates, risks, or business conclusions.
- Keep the answer clear and business-friendly.

Your Execution architecture:
Select Tool
    |
Retrieve Policy Context
    |
Grade Context: if the context is irrelevant call the required tool
    |
Generate Final Answer

NOTE:
Do not rewrite the query more than once if the context is not matched.

Question:
{user_query}

Retrieved Context:
{context}


Return:
1. Answer
2. Supporting Evidence
3. Sources
4. Confidence: High / Medium / Low

"""

In [ ]:
from langchain.tools.retriever import create_retriever_tool

In [ ]:
retrieve_hr_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_hr_policy",
    description="Search and return information about hr policy."
)

In [ ]:
retrieve_travel_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_travel_policy",
    description="Search and return information about travel policy."
)

In [ ]:
retrieve_reimbursement_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_reimbursement_policy",
    description="Search and return information about reimbursement policy."
)

In [ ]:
retrieve_it_security_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_it_security_policy",
    description="Search and return information about IT security policy."
)

In [ ]:
retrieve_ai_usage_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_ai_usage_policy",
    description="Search and return information about ai usage policy."
)

In [ ]:

def grade_retrieved_context(context, user_query):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        Retrived Context (context): The context rerieved by retriever tool.

    Returns:
        str: A decision for whether the documents are relevant or not
    """
    print("---CHECK RELEVANCE---")


In [ ]:
def rewrite_query(grader_output, user_query):
    """
    Transform the user query to produce a better question.

    Args:
        Grader's output: 

    Returns:
        dict: The updated state with re-phrased question
    """
    

In [ ]:
tools = [retrieve_ai_usage_policy, retrieve_it_security_policy, retrieve_reimbursement_policy, retrieve_travel_policy, retrieve_hr_policy, grade_retrieved_context]

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

In [ ]:
agent.invoke(". What should I do if the policy does not mention my scenario?")